In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_openml

In [2]:
data_class = fetch_openml(name='adult', version=2, as_frame=True)
df_income = data_class.frame
df_income

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,NaN,103497,Some-college,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
48838,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
48839,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
48840,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [ ]:
X = df_income.drop('class', axis=1)
y = (df_income['class'] == '>50K').astype(int)
for col in X.select_dtypes(include='category').columns:
    X[col] = X[col].astype(str)
    X[col] = pd.Categorical(X[col]).codes


X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.25)

clf = XGBClassifier(
    eval_metric='logloss',
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    random_state=0, #params for train accuracy= 87.863% test accuracy= 87.315%, little overfitting
)
clf.fit(X_train, y_train)

print("train accuracy= {:.3%}".format(clf.score(X_train, y_train)))
print("test accuracy= {:.3%}".format(clf.score(X_test, y_test)))

train accuracy= 87.863%
test accuracy= 87.315%


In [ ]:
from xgboost import XGBRegressor

data_reg = fetch_openml(name='house_prices', version=1, as_frame=True)
df_house = data_reg.frame
df_house

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,NaN,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,142125


In [ ]:
X = df_house.drop('SalePrice', axis=1)
y = df_house['SalePrice'].astype(float)

# encode all text/category columns to integer codes
for col in X.select_dtypes(include=['category', 'object', 'str']).columns:
    X[col] = pd.Categorical(X[col]).codes

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, test_size=0.25)

reg = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.01,
    min_child_weight=5,
    random_state=0, #train 0.907 test 0.841, little overfitting(0.066)
)
reg.fit(X_train, y_train)

print("train = {:.3f}".format(reg.score(X_train, y_train)))
print("test  = {:.3f}".format(reg.score(X_test, y_test)))

train = 0.907
test  = 0.841


In [ ]:
pd.Series(reg.feature_importances_, index=X.columns).sort_values(ascending=False).head(10) #importance of features

OverallQual    0.293626
GarageCars     0.199298
BsmtQual       0.077061
GrLivArea      0.035406
GarageType     0.024237
CentralAir     0.020299
BsmtFinSF1     0.017610
YearBuilt      0.015858
1stFlrSF       0.015378
Fireplaces     0.015153
dtype: float32